# Complex impedance on a Smith chart

A small Quantity subclass makes circuit expressions readable: addition connects
impedances in series and floor division connects them in parallel. Its implementation
lives once in the companion [impedance.py](impedance.py); keep the two files together.

This illustrative calculation uses ideal lumped components and RMS phasors with the
convention $e^{i\omega t}$. Every branch in one network is evaluated at the same
frequency. The class stores an impedance value, without storing frequency or topology.

$$
Z_R=R,\qquad Z_L=i\omega L,\qquad Z_C=\frac{1}{i\omega C},\qquad
Z_1\parallel Z_2=\frac{Z_1Z_2}{Z_1+Z_2}.
$$

We sweep a series RLC branch, then place a resistor in parallel with it.
The 50-ohm reference sets the Smith-chart normalization; it is not an additional
physical resistor in either network.


In [ ]:
import math
from fractions import Fraction

import numpy as np
from impedance import Impedance
from matplotlib import pyplot as plt
from matplotlib.axes import Axes

from unitx import Quantity
from unitx import math as unit_math
from unitx.si import si

## Start with a resistor network

Real impedances describe ordinary resistors. Fraction magnitudes preserve exact
rational results. Series and parallel composition and unit conversion return
Impedance; voltage and current calculations return ordinary Quantity.


In [ ]:
first = Impedance(Fraction(100), si.ohm)
second = Impedance(Fraction(1, 5), si.kohm)
third = Impedance(Fraction(300), si.ohm)

resistor_network = (first + second).to(si.kohm) // third
assert isinstance(resistor_network, Impedance)
assert resistor_network == 150 * si.ohm
assert first // second // third == Fraction(600, 11) * si.ohm
assert Impedance(0) // Impedance(100j) == 0 * si.ohm
print(f'({first} + {second.to(first.unit)}) // ({third}) = {resistor_network}')

## Evaluate the circuit over frequency

Choose $R=25\,\Omega$, $L=1\,\mathrm{mH}$ and $C=1\,\mathrm{\mu F}$.
The additional parallel branch has impedance $200\,\Omega$.

The extension intentionally holds scalar values. We evaluate one frequency at a
time, then assemble the results into one array Quantity with a common unit.
Physical calculations retain their units until the explicit constructor boundary.
These are illustrative component values, not measured data; parasitic resistance,
distributed effects and transmission-line propagation are outside the model.


In [ ]:
resistance = 25 * si.ohm
inductance = 1 * si.mH
capacitance = 1 * si.uF
shunt = Impedance(200, si.ohm)
reference = 50 * si.ohm
frequency = Quantity(np.geomspace(100, 50_000, 600), si.Hz)


def series_at(frequency: Quantity[float]) -> Impedance:
    omega = 2 * math.pi * frequency
    reactance = omega * inductance - 1 / (omega * capacitance)
    return Impedance(
        complex(float(resistance.value_in(si.ohm)), float(reactance.value_in(si.ohm))),
        si.ohm,
    )


series_samples = [series_at(Quantity(float(value), si.Hz)) for value in frequency.value]
series_impedance = Quantity(
    np.asarray([value.value_in(si.ohm) for value in series_samples], dtype=np.complex128),
    si.ohm,
)
parallel_impedance = Quantity(
    np.asarray(
        [(value // shunt).value_in(si.ohm) for value in series_samples], dtype=np.complex128
    ),
    si.ohm,
)
assert isinstance(series_impedance, Quantity) and isinstance(parallel_impedance, Quantity)
assert series_impedance.unit == parallel_impedance.unit == si.ohm
assert np.all(series_impedance.value.real > 0) and np.all(parallel_impedance.value.real > 0)

## Normalize and check the physical picture

For a real positive reference $Z_0$, the reflection coefficient is

$$
z=\frac{Z}{Z_0},\qquad \Gamma=\frac{z-1}{z+1}=\frac{Z-Z_0}{Z+Z_0}.
$$

The centre of the Smith chart is a match, $Z=Z_0$. A short circuit is at
$\Gamma=-1$ and an open circuit approaches $\Gamma=+1$. Passive impedances lie
inside or on the unit disk. Positive reactance occupies the upper half and negative
reactance the lower half under the convention used here.

At $f_0=1/(2\pi\sqrt{LC})$, the inductor and capacitor cancel in the series branch.
Its impedance is then 25 ohm; the parallel network is $200/9$ ohm. The exact
resonance reflection coefficients are $-1/3$ and $-5/13$, respectively.


In [ ]:
gamma_series = np.asarray(
    ((series_impedance - reference) / (series_impedance + reference)).value_in(si.one),
    dtype=np.complex128,
)
gamma_parallel = np.asarray(
    ((parallel_impedance - reference) / (parallel_impedance + reference)).value_in(si.one),
    dtype=np.complex128,
)
assert isinstance(gamma_series, np.ndarray) and isinstance(gamma_parallel, np.ndarray)
assert np.all(np.isfinite(gamma_series)) and np.all(np.isfinite(gamma_parallel))
assert np.all(np.abs(gamma_series) <= 1 + 1e-12)
assert np.all(np.abs(gamma_parallel) <= 1 + 1e-12)

resonance_frequency = (1 / (2 * math.pi * unit_math.sqrt(inductance * capacitance))).to(si.Hz)
resonance_series = series_at(Quantity(float(resonance_frequency.value), si.Hz))
resonance_parallel = resonance_series // shunt
np.testing.assert_allclose(complex(resonance_series.value), 25, atol=1e-10)
np.testing.assert_allclose(complex(resonance_parallel.value), 200 / 9, atol=1e-10)


def reflection(impedance: Impedance) -> complex:
    normalized = impedance.value_in(si.ohm) / reference.value_in(si.ohm)
    return complex((normalized - 1) / (normalized + 1))


resonance_gamma_series = reflection(resonance_series)
resonance_gamma_parallel = reflection(resonance_parallel)
np.testing.assert_allclose(resonance_gamma_series, -1 / 3, atol=1e-12)
np.testing.assert_allclose(resonance_gamma_parallel, -5 / 13, atol=1e-12)

voltage = 12 * si.V  # RMS phasor at resonance.
current = voltage / resonance_parallel
current_amperes = current.value_in(si.A)
assert isinstance(current_amperes, complex)
np.testing.assert_allclose(current_amperes, 0.54, atol=1e-12)
print(f'Resonance: {resonance_frequency.to(si.kHz):.3f}')
print(f'At {voltage} RMS: {current.to(si.mA):.3f}')

## Draw the Smith chart

The grey grid consists of constant normalized resistance circles and constant
normalized reactance arcs. Both trajectories use the same 50-ohm reference.
Arrows and the labelled points indicate increasing frequency. The square marks
the parallel network at resonance; the circle marks the series branch.
Plotting uses the explicit unitless reflection coefficients.


In [ ]:
def draw_smith_grid(ax: Axes) -> None:
    theta = np.linspace(0, 2 * np.pi, 900)
    grid_color = '#cbd5e1'
    for r in (0.2, 0.5, 1, 2, 5):
        centre, radius = r / (1 + r), 1 / (1 + r)
        ax.plot(
            centre + radius * np.cos(theta),
            radius * np.sin(theta),
            color=grid_color,
            linewidth=0.7,
            zorder=0,
        )
        ax.text(
            (r - 1) / (r + 1), -0.035, f'{r:g}', ha='center', va='top', fontsize=8, color='#64748b'
        )
    r_values = np.concatenate(([0.0], np.geomspace(1e-4, 1e5, 600)))
    for x in (0.2, 0.5, 1, 2, 5):
        for sign in (-1, 1):
            z = r_values + sign * 1j * x
            gamma = (z - 1) / (z + 1)
            ax.plot(gamma.real, gamma.imag, color=grid_color, linewidth=0.7, zorder=0)
            edge = (sign * 1j * x - 1) / (sign * 1j * x + 1)
            ax.text(
                1.065 * edge.real,
                1.065 * edge.imag,
                f'{"+" if sign > 0 else "-"}j{x:g}',
                ha='center',
                va='center',
                fontsize=8,
                color='#64748b',
            )
    ax.plot(np.cos(theta), np.sin(theta), color='#475569', linewidth=1.1)
    ax.axhline(0, color='#94a3b8', linewidth=0.8, zorder=0)
    ax.plot(0, 0, '.', color='#475569', markersize=4)
    ax.text(0.025, 0.025, 'match', fontsize=8, color='#64748b')
    ax.text(-1.11, 0.02, 'short', ha='center', fontsize=9, color='#475569')
    ax.text(1.1, 0.02, 'open', ha='center', fontsize=9, color='#475569')
    ax.text(0, 1.13, 'inductive (+jX)', ha='center', fontsize=10, color='#475569')
    ax.text(0, -1.13, 'capacitive (-jX)', ha='center', fontsize=10, color='#475569')
    ax.set(xlim=(-1.22, 1.22), ylim=(-1.22, 1.22), aspect='equal')
    ax.axis('off')


fig, ax = plt.subplots(figsize=(8.4, 8.4), layout='constrained')
draw_smith_grid(ax)
blue, orange = '#2563eb', '#ea580c'
ax.plot(gamma_series.real, gamma_series.imag, color=blue, linewidth=2.2, label='Series RLC')
ax.plot(
    gamma_parallel.real,
    gamma_parallel.imag,
    color=orange,
    linewidth=2.2,
    label=rf'RLC $\parallel {shunt.to_latex()}$',
)

arrow_frequency = 9 * si.kHz
for trajectory, color in ((gamma_series, blue), (gamma_parallel, orange)):
    index = int(
        np.argmin(np.abs(frequency.value - float(arrow_frequency.value_in(frequency.unit))))
    )
    start, end = trajectory[index], trajectory[index + 12]
    ax.annotate(
        '',
        xy=(end.real, end.imag),
        xytext=(start.real, start.imag),
        arrowprops={'arrowstyle': '->', 'color': color, 'lw': 2},
    )

for target, offset in (
    (100 * si.Hz, (-65, -30)),
    (2 * si.kHz, (15, -30)),
    (20 * si.kHz, (-35, 23)),
    (50 * si.kHz, (22, -15)),
):
    index = int(np.argmin(np.abs(frequency.value - float(target.value_in(frequency.unit)))))
    point = gamma_parallel[index]
    ax.plot(point.real, point.imag, 'o', color=orange, markersize=4)
    ax.annotate(
        str(target),
        (point.real, point.imag),
        xytext=offset,
        textcoords='offset points',
        fontsize=9,
        color='#9a3412',
        arrowprops={'arrowstyle': '-', 'color': orange, 'lw': 0.7},
    )

ax.plot(resonance_gamma_series.real, resonance_gamma_series.imag, 'o', color=blue, markersize=6)
ax.plot(
    resonance_gamma_parallel.real, resonance_gamma_parallel.imag, 's', color=orange, markersize=6
)
ax.annotate(
    f'Resonance: {resonance_frequency.to(si.kHz):.3f}',
    (resonance_gamma_parallel.real, resonance_gamma_parallel.imag),
    xytext=(-0.91, -0.48),
    fontsize=10,
    color='#334155',
    arrowprops={'arrowstyle': '->', 'color': '#64748b', 'lw': 0.9},
)
ax.legend(loc='lower left', frameon=False, fontsize=10)
ax.set_title(
    'RLC impedance on a Smith chart\n'
    f'$Z_0={reference.to_latex()}$  |  '
    f'$R={resistance.to_latex()}$, '
    f'$L={inductance.to_latex()}$, '
    f'$C={capacitance.to_latex()}$',
    fontsize=13,
    pad=8,
)
plt.show()